# CoT Enrichment v2 — Quality Audit

Inspect both the **private native `<think>` block** (model's deliberation, discarded for SFT) and the **visible `<reasoning>` block** (SFT-target trace) for each enriched example.

Checks:
1. Top-line stats — extraction source, answer-match rate, length distribution
2. Native think extraction — does the model leak gold here? (private, not in SFT)
3. Reasoning leak check — does the SFT-target trace reference gold? (would poison SFT)
4. Parser robustness flags — caught any malformed `<reasoning>` blocks?
5. Per-example side-by-side viewer
6. Mismatch deep-dive — candidate vote distribution for non-match ids

In [ ]:
import json, re, random
from pathlib import Path
from collections import Counter, defaultdict

OUT = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill/teacher_enriched_1058579_TEST.jsonl')
STAGE_A = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill/cot_1058163_Qwen3-VL-235B-A22B-Thinking-FP8_train_N16_T0.8_grounding.jsonl')

rows = [json.loads(l) for l in OUT.open()]
print(f'enriched rows:  {len(rows)}')
print(f'parse_ok:       {sum(r["parse_ok"] for r in rows)}/{len(rows)}')

# Stage A: gold answers, question text, per-id candidate answers
gold = {}
qmap = {}
cand_answers = defaultdict(list)
for line in STAGE_A.open():
    if not line.endswith('\n'):
        break
    s = json.loads(line)
    if s['id'] not in gold:
        gold[s['id']] = s['gt_answer']
        qmap[s['id']] = s['prompt']
    cand_answers[s['id']].append((s.get('answer') or '').strip())
print(f'stage A ids loaded: {len(gold)}')

## 1. Top-line stats

In [ ]:
src_count = Counter(r['parsed']['reasoning_source'] for r in rows if r['parse_ok'])
empty_count = sum(1 for r in rows if r['parse_ok'] and not r['parsed']['reasoning'])
match_count = 0
mismatch = []
for r in rows:
    if not r['parse_ok']:
        continue
    g = gold.get(r['id'], '').strip().lower()
    a = r['parsed']['answer'].strip().lower()
    if g and a == g:
        match_count += 1
    else:
        mismatch.append(r['id'])

reasoning_lens = [len(r['parsed']['reasoning']) for r in rows if r['parse_ok']]
print(f"reasoning_source distribution: {dict(src_count)}")
print(f"reasoning empty:               {empty_count}/{len(rows)}")
print(f"answer matches gold:           {match_count}/{len(rows)}")
if reasoning_lens:
    print(f"reasoning length: min={min(reasoning_lens)} max={max(reasoning_lens)} mean={sum(reasoning_lens)//len(reasoning_lens)} chars")
print(f"mismatch ids: {mismatch}")

## 2. Native `<think>` extraction

The chat template injects opening `<think>` as a special token; only `</think>` appears in `raw`. Native think = everything from the start of generation up to the first `</think>`. This is **private deliberation** — never used for SFT — but worth auditing to see how the model deliberates and whether it cites gold privately.

In [ ]:
def extract_native_think(raw: str) -> str:
    """Native think: text before the first </think>. Strip leading 'thinking' literal
    that the chat template's special <think> token sometimes decodes to."""
    if '</think>' not in raw:
        return raw.strip()
    pre = raw.split('</think>', 1)[0]
    return re.sub(r'^\s*thinking\s*', '', pre).strip()

LEAK_RE = re.compile(
    r'\bgold answer\b|\bgiven (the|that the) answer\b|'
    r'\bsince the answer is\b|\bworking backward\b|\bwe know the answer\b',
    re.IGNORECASE,
)

think_lens = []
think_leaks = 0
reason_leaks = []
for r in rows:
    if not r['parse_ok']:
        continue
    nt = extract_native_think(r['raw'])
    think_lens.append(len(nt))
    if LEAK_RE.search(nt):
        think_leaks += 1
    if LEAK_RE.search(r['parsed']['reasoning']):
        reason_leaks.append(r['id'])

print(f"native think length: min={min(think_lens)} max={max(think_lens)} mean={sum(think_lens)//len(think_lens)} chars")
print(f"native <think> mentions gold/answer:  {think_leaks}/{len(rows)}  (PRIVATE — fine if non-zero)")
print(f"<reasoning> mentions gold/answer:     {len(reason_leaks)}/{len(rows)}  (BAD — would leak into SFT)  ids={reason_leaks}")

## 3. Parser robustness flags

Catch cases where the regex captured a malformed `<reasoning>` block — e.g. it contains a literal `</think>` (means the regex glued native think + visible reasoning), or contains tag literals (means the model emitted nested or stray tags).

In [ ]:
parser_issues = []
for r in rows:
    if not r['parse_ok']:
        continue
    rsn = r['parsed']['reasoning']
    issues = []
    if '</think>' in rsn:
        issues.append('contains </think>')
    if '<reasoning>' in rsn:
        issues.append('contains <reasoning> open literal')
    if '</reasoning>' in rsn:
        issues.append('contains </reasoning> close literal')
    n_answer_lines = sum(1 for line in rsn.splitlines() if line.lower().lstrip().startswith('answer:'))
    if n_answer_lines > 1:
        issues.append(f'has {n_answer_lines} "Answer:" lines')
    if issues:
        parser_issues.append((r['id'], r['parsed']['reasoning_source'], issues))

print(f"parser issues: {len(parser_issues)}/{len(rows)}")
for id_, src, iss in parser_issues:
    print(f"  [{id_}] source={src} -- {', '.join(iss)}")

## 4. Per-example side-by-side viewer

For each id: question · gold · answer (match?) · native think · visible reasoning.

In [ ]:
def show(r, max_think_chars=4000):
    p = r['parsed']
    g = gold.get(r['id'], '').strip()
    a = p['answer'].strip()
    match = 'MATCH' if a.lower() == g.lower() else 'MISMATCH'
    nt = extract_native_think(r['raw'])
    print('=' * 100)
    print(f"ID: {r['id']}   [{match}]   parse_ok={r['parse_ok']}  source={p.get('reasoning_source')}")
    print(f"QUESTION: {qmap.get(r['id'], '?')[:400].strip()}")
    print(f"GOLD:    {g}")
    print(f"ANSWER:  {a}")
    print(f"\n----- NATIVE <think> ({len(nt)} chars, PRIVATE — discarded) -----")
    if len(nt) > max_think_chars:
        cut = max_think_chars // 2
        print(nt[:cut] + f"\n\n  [... {len(nt)-max_think_chars} chars cut ...]\n\n" + nt[-cut:])
    else:
        print(nt)
    print(f"\n----- VISIBLE <reasoning> ({len(p['reasoning'])} chars, SFT TARGET) -----")
    print(p['reasoning'])
    print()

In [ ]:
# Show all records
for r in rows:
    show(r)

## 5. Mismatch deep-dive — candidate vote breakdown

For each id where teacher answer != gold, show how the 16 Stage-A candidates voted.
If 12+/16 candidates landed on the teacher's (non-gold) answer, the visual evidence honestly contradicts gold — these are likely Stage A label noise, not teacher errors.

In [ ]:
for r in rows:
    if not r['parse_ok']:
        continue
    g = gold.get(r['id'], '').strip()
    a = r['parsed']['answer'].strip()
    if a.lower() == g.lower():
        continue
    cands = cand_answers[r['id']]
    c = Counter(cands)
    print('=' * 80)
    print(f"id={r['id']}  GOLD={g!r}  TEACHER={a!r}")
    for ans, k in c.most_common():
        flag = ''
        if ans.lower() == g.lower():
            flag = '  <-- gold'
        elif ans.lower() == a.lower():
            flag = '  <-- teacher'
        disp = (ans[:80] + '...') if len(ans) > 80 else ans
        print(f"  {k:2d}x {disp!r}{flag}")

---

# Stage B — Qwen-as-judge audit (job 1058608, first 4 ids)

Per-id viewer for the new pipeline:
- 16 candidate score vectors (hallucination / visual_grounding / reasoning_quality / answer_correctness)
- winner highlighted with a `*`
- `scene_description` (B2 output, native `<think>` already stripped by `qwen_judge.py`)
- raw winner trace + grounding for spot-check

In [ ]:
JUDGE_OUT = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill/judge_1058608_qwen235b.jsonl')

judge_rows = [json.loads(l) for l in JUDGE_OUT.open()]
print(f'judge rows: {len(judge_rows)}')

# Stage A samples keyed by id -> sample_idx for winner-trace lookup
stage_a_by_id = defaultdict(dict)
for line in Path('/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill/cot_1058163_Qwen3-VL-235B-A22B-Thinking-FP8_train_N16_T0.8_grounding.jsonl').open():
    if not line.endswith('\n'):
        break
    s = json.loads(line)
    stage_a_by_id[s['id']][s['sample_idx']] = s

def show_judge(r, n_winner_trace_chars=2000):
    id_ = r['id']
    g = stage_a_by_id[id_][0]['gt_answer']
    q = stage_a_by_id[id_][0]['prompt']
    best_idx = r['best_idx']
    best_total = r['best_score_total']
    print('=' * 100)
    print(f"ID: {id_}   best_idx={best_idx}   best_total={best_total}   "
          f"all_b1_failed={r['all_b1_failed']}   parse_ok_b2={r['parse_ok_b2']}")
    print(f"QUESTION: {q[:400].strip()}")
    print(f"GOLD:    {g}")
    # Score table
    print(f"\n----- 16-CANDIDATE SCORES -----")
    print(f"{'idx':>3} {'hal':>3} {'vg':>3} {'rq':>3} {'ac':>2} {'tot':>3}  cand_answer")
    for i, s in enumerate(r['scores']):
        marker = '*' if i == best_idx else ' '
        if s is None:
            cand_ans = (stage_a_by_id[id_][i].get('answer') or '').strip()
            print(f"{marker}{i:>2} PARSE_FAIL                     {cand_ans[:60]!r}")
            continue
        cand_ans = (stage_a_by_id[id_][i].get('answer') or '').strip()
        print(f"{marker}{i:>2} "
              f"{s['hallucination']['score']:>3} "
              f"{s['visual_grounding']['score']:>3} "
              f"{s['reasoning_quality']['score']:>3} "
              f"{s['answer_correctness']:>2} "
              f"{s['score_total']:>3}  {cand_ans[:60]!r}")
    # Scene description
    print(f"\n----- SCENE DESCRIPTION (B2, {len(r['scene_description'])} chars) -----")
    print(r['scene_description'])
    # Winner trace + grounding
    if not r['all_b1_failed']:
        winner = stage_a_by_id[id_][best_idx]
        print(f"\n----- WINNER GROUNDING -----")
        print(winner.get('grounding', '').strip())
        print(f"\n----- WINNER TRACE ({len(winner.get('thinking') or '')} chars) -----")
        wt = winner.get('thinking', '') or ''
        if len(wt) > n_winner_trace_chars:
            print(wt[:n_winner_trace_chars] + f"\n  [... cut {len(wt)-n_winner_trace_chars} chars ...]")
        else:
            print(wt)
    print()

for r in judge_rows[:4]:
    show_judge(r)

In [ ]:
# Stage B (Qwen3-VL-32B-Thinking judge) — job 1058653, all 10 smoke-test ids
# Reuses show_judge() defined in the cell above. ~2.6x faster than 235B but
# more lenient (38.7% candidates score 20 vs 27.2%).

JUDGE_OUT_32B = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill/judge_1058653_qwen32b.jsonl')
judge_rows_32b = [json.loads(l) for l in JUDGE_OUT_32B.open()]
print(f'judge rows: {len(judge_rows_32b)}\n')

for r in judge_rows_32b:
    show_judge(r)